# Credit Card Transaction Analysis & Fraud Detection
## Stage 1: Data Preprocessing and Feature Engineering

In this notebook, we load the raw `fraudTrain.csv` and `fraudTest.csv` datasets, verify their shapes and data quality, extract temporal and demographic features, calculate spatial distance between cardholders and merchants, categorize transaction amounts, drop raw identifiers, and export clean datasets.

### Step 1: Import Libraries and Load Datasets

In [ ]:
import os
import numpy as np
import pandas as pd

train_path = os.path.join("..", "data", "fraudTrain.csv")
test_path = os.path.join("..", "data", "fraudTest.csv")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print("Training data shape:", train.shape)
print("Testing data shape:", test.shape)
print("\nFirst 5 rows of training data:")
train.head()

### Step 2: Check Info, Missing Values, and Duplicates

In [ ]:
print("--- Training Data Info ---")
train.info()

print("\n--- Missing Values in Train ---")
print(train.isnull().sum())

print("\n--- Duplicates in Train ---")
print(train.duplicated().sum())

print("\n--- Duplicates in Test ---")
print(test.duplicated().sum())

### Step 3: Datetime Feature Engineering

In [ ]:
for df in [train, test]:
    df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"])
    df["transaction_hour"] = df["trans_date_trans_time"].dt.hour
    df["day_of_week"] = df["trans_date_trans_time"].dt.dayofweek
    df["month"] = df["trans_date_trans_time"].dt.month
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

train[["trans_date_trans_time", "transaction_hour", "day_of_week", "month", "is_weekend"]].head()

### Step 4: Customer Age Calculation

In [ ]:
for df in [train, test]:
    df["dob"] = pd.to_datetime(df["dob"])
    df["customer_age"] = df["trans_date_trans_time"].dt.year - df["dob"].dt.year

train[["dob", "customer_age"]].head()

### Step 5: Haversine Merchant Distance Calculation

In [ ]:
def calculate_distance(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371 * c

for df in [train, test]:
    df["merchant_distance"] = calculate_distance(
        df["lat"], df["long"], df["merch_lat"], df["merch_long"]
    )

train[["lat", "long", "merch_lat", "merch_long", "merchant_distance"]].head()

### Step 6: Amount Categorization

In [ ]:
for df in [train, test]:
    df["amount_category"] = pd.cut(
        df["amt"],
        bins=[0, 50, 100, 500, 1000, float("inf")],
        labels=["Low", "Medium", "High", "Very High", "Extreme"]
    )

train[["amt", "amount_category"]].head()

### Step 7: Dropping Unnecessary & Identifier Columns

In [ ]:
drop_columns = [
    "Unnamed: 0",
    "first",
    "last",
    "street",
    "trans_num",
    "dob",
    "cc_num",
    "unix_time",
    "lat",
    "long",
    "merch_lat",
    "merch_long"
]

train_clean = train.drop(columns=[c for c in drop_columns if c in train.columns])
test_clean = test.drop(columns=[c for c in drop_columns if c in test.columns])

print("Cleaned Train shape:", train_clean.shape)
print("Cleaned Test shape:", test_clean.shape)
train_clean.head()

### Step 8: Save Cleaned Datasets

In [ ]:
os.makedirs(os.path.join("..", "data", "processed"), exist_ok=True)

train_clean.to_csv(os.path.join("..", "data", "processed", "fraudTrain_clean.csv"), index=False)
test_clean.to_csv(os.path.join("..", "data", "processed", "fraudTest_clean.csv"), index=False)

print("Successfully saved processed datasets!")